# Intent Classifier v2 - Direct Colab Training

This notebook trains the intent classifier directly in notebook cells.

- No `Trainer`
- No `pipeline`
- No full project upload
- Upload only `colab_intent_v2_bundle.zip`

It retrains only the intent classifier. It does not touch the NER BERT+CRF+Viterbi model.

## 1. Upload Bundle

In [1]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

Saving colab_intent_v2_bundle.zip to colab_intent_v2_bundle (1).zip
dict_keys(['colab_intent_v2_bundle (1).zip'])


## 2. Unzip Workspace

In [2]:
import os

zip_names = [name for name in uploaded.keys() if name.endswith('.zip')]
assert zip_names, 'Please upload colab_intent_v2_bundle.zip'
zip_name = zip_names[0]

!rm -rf /content/intent_v2_workspace
!mkdir -p /content/intent_v2_workspace
!unzip -q "{zip_name}" -d /content/intent_v2_workspace

PROJECT_DIR = '/content/intent_v2_workspace/colab_intent_v2_bundle'
os.chdir(PROJECT_DIR)
print('Current dir:', os.getcwd())
!find . -maxdepth 3 -type f | sort

Current dir: /content/intent_v2_workspace/colab_intent_v2_bundle
./data/en/intent_data.csv
./data/en/intent_hard_review.csv
./notebooks/en/train_intent_bert_v2_colab.ipynb
./notebooks/en/train_intent_bert_v2_direct_colab.ipynb
./notebooks/en/train_intent_bert_v2_upload_colab.ipynb
./reports/en/intent_dataset_v2_report.md
./reports/en/intent_v2_baseline.md
./reports/en/intent_v2_retrain_plan.md
./scripts/build_intent_v2_dataset.py
./scripts/evaluate_intent_model.py
./scripts/train_intent_bert_v2.py


## 3. Install Dependencies

`torchvision` is removed because some Colab images have a mismatched torchvision build that breaks text-only Transformers imports.

In [3]:
!pip uninstall -y torchvision
!pip install -U "transformers==4.44.2" scikit-learn pandas torch safetensors

## 4. Check GPU

In [4]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA L4


## 5. Build Dataset v2

In [5]:
!python scripts/build_intent_v2_dataset.py

Saved hard-set: data/en/intent_hard_balanced_review.csv (300 rows)
Saved train: data/en/intent_v2/intent_train_v2.csv (1680 rows)
Saved val: data/en/intent_v2/intent_hard_val.csv (60 rows)
Saved test: data/en/intent_v2/intent_hard_test.csv (60 rows)


In [6]:
import pandas as pd

for path in [
    'data/en/intent_v2/intent_train_v2.csv',
    'data/en/intent_v2/intent_hard_val.csv',
    'data/en/intent_v2/intent_hard_test.csv',
    'data/en/intent_v2/intent_hotpot_relabel_test.csv',
]:
    df = pd.read_csv(path)
    print('\n', path, df.shape)
    print(df['label'].value_counts())


 data/en/intent_v2/intent_train_v2.csv (1680, 2)
label
NUTRITION_LOOKUP    560
BOTH                560
HEALTH_ADVICE       560
Name: count, dtype: int64

 data/en/intent_v2/intent_hard_val.csv (60, 2)
label
NUTRITION_LOOKUP    20
HEALTH_ADVICE       20
BOTH                20
Name: count, dtype: int64

 data/en/intent_v2/intent_hard_test.csv (60, 2)
label
HEALTH_ADVICE       20
NUTRITION_LOOKUP    20
BOTH                20
Name: count, dtype: int64

 data/en/intent_v2/intent_hotpot_relabel_test.csv (71, 2)
label
HEALTH_ADVICE    71
Name: count, dtype: int64


## 6. Direct Training Code

In [7]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABELS = ['NUTRITION_LOOKUP', 'HEALTH_ADVICE', 'BOTH']
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

OUTPUT_DIR = Path('models/classifier_bert_v2')
REPORT_DIR = Path('reports/en/intent_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [8]:
def load_csv(path):
    df = pd.read_csv(path)
    assert set(df.columns) == {'text', 'label'}, (path, df.columns)
    return df.dropna(subset=['text', 'label']).reset_index(drop=True)


train_all = load_csv('data/en/intent_v2/intent_train_v2.csv')
hard_val = load_csv('data/en/intent_v2/intent_hard_val.csv')
hard_test = load_csv('data/en/intent_v2/intent_hard_test.csv')
hotpot_test = load_csv('data/en/intent_v2/intent_hotpot_relabel_test.csv')

train_df, synthetic_val = train_test_split(
    train_all,
    test_size=0.1,
    random_state=SEED,
    stratify=train_all['label'],
)

print('train:', train_df.shape, train_df['label'].value_counts().to_dict())
print('synthetic_val:', synthetic_val.shape, synthetic_val['label'].value_counts().to_dict())
print('hard_val:', hard_val.shape, hard_val['label'].value_counts().to_dict())
print('hard_test:', hard_test.shape, hard_test['label'].value_counts().to_dict())
print('hotpot_test:', hotpot_test.shape, hotpot_test['label'].value_counts().to_dict())

train: (1512, 2) {'BOTH': 504, 'NUTRITION_LOOKUP': 504, 'HEALTH_ADVICE': 504}
synthetic_val: (168, 2) {'HEALTH_ADVICE': 56, 'BOTH': 56, 'NUTRITION_LOOKUP': 56}
hard_val: (60, 2) {'NUTRITION_LOOKUP': 20, 'HEALTH_ADVICE': 20, 'BOTH': 20}
hard_test: (60, 2) {'HEALTH_ADVICE': 20, 'NUTRITION_LOOKUP': 20, 'BOTH': 20}
hotpot_test: (71, 2) {'HEALTH_ADVICE': 71}


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def make_dataset(df):
    enc = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    labels = torch.tensor([LABEL2ID[label] for label in df['label'].tolist()], dtype=torch.long)
    return TensorDataset(enc['input_ids'], enc['attention_mask'], labels)


train_loader = DataLoader(make_dataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
synthetic_val_loader = DataLoader(make_dataset(synthetic_val), batch_size=BATCH_SIZE)
hard_val_loader = DataLoader(make_dataset(hard_val), batch_size=BATCH_SIZE)
hard_test_loader = DataLoader(make_dataset(hard_test), batch_size=BATCH_SIZE)
hotpot_loader = DataLoader(make_dataset(hotpot_test), batch_size=BATCH_SIZE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def evaluate(loader, name):
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0.0

    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += float(out.loss.detach().cpu())

            preds = torch.argmax(out.logits, dim=-1)
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(preds.detach().cpu().tolist())

    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(LABELS))),
        target_names=LABELS,
        output_dict=True,
        zero_division=0,
    )

    return {
        "name": name,
        "loss": total_loss / max(len(loader), 1),
        "accuracy": float(acc),
        "macro_f1": float(macro),
        "confusion_matrix": matrix.tolist(),
        "classification_report": report,
        "labels": LABELS,
    }
    return result


def save_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, indent=2), encoding='utf-8')


best_hard_val_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for step, (input_ids, attention_mask, labels) in enumerate(train_loader, start=1):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        out.loss.backward()
        optimizer.step()
        losses.append(float(out.loss.detach().cpu()))

        if step % 20 == 0:
            print(f'Epoch {epoch}/{EPOCHS} step {step}/{len(train_loader)} loss={np.mean(losses[-20:]):.4f}')

    syn = evaluate(synthetic_val_loader, 'synthetic_val')
    hv = evaluate(hard_val_loader, 'hard_val')
    print(f"Epoch {epoch}: train_loss={np.mean(losses):.4f} synthetic_f1={syn['macro_f1']:.4f} hard_val_f1={hv['macro_f1']:.4f}")
    print('hard_val confusion:', hv['confusion_matrix'])

    if hv['macro_f1'] > best_hard_val_f1:
        best_hard_val_f1 = hv['macro_f1']
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print('Saved best model to', OUTPUT_DIR)

print('Best hard_val macro-F1:', best_hard_val_f1)

Epoch 1/3 step 20/95 loss=0.0095
Epoch 1/3 step 40/95 loss=0.0096
Epoch 1/3 step 60/95 loss=0.0093
Epoch 1/3 step 80/95 loss=0.0092
Epoch 1: train_loss=0.0094 synthetic_f1=1.0000 hard_val_f1=1.0000
hard_val confusion: [[20, 0, 0], [0, 20, 0], [0, 0, 20]]
Saved best model to models/classifier_bert_v2
Epoch 2/3 step 20/95 loss=0.0099
Epoch 2/3 step 40/95 loss=0.0092
Epoch 2/3 step 60/95 loss=0.0097
Epoch 2/3 step 80/95 loss=0.0093
Epoch 2: train_loss=0.0095 synthetic_f1=1.0000 hard_val_f1=1.0000
hard_val confusion: [[20, 0, 0], [0, 20, 0], [0, 0, 20]]
Epoch 3/3 step 20/95 loss=0.0095
Epoch 3/3 step 40/95 loss=0.0094
Epoch 3/3 step 60/95 loss=0.0097
Epoch 3/3 step 80/95 loss=0.0092
Epoch 3: train_loss=0.0094 synthetic_f1=1.0000 hard_val_f1=1.0000
hard_val confusion: [[20, 0, 0], [0, 20, 0], [0, 0, 20]]
Best hard_val macro-F1: 1.0


## 7. Final Evaluation

In [13]:
# Load best saved model before final evaluation.
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.to(device)

synthetic_result = evaluate(synthetic_val_loader, 'synthetic_val')
hard_val_result = evaluate(hard_val_loader, 'hard_val')
hard_test_result = evaluate(hard_test_loader, 'hard_test')
hotpot_result = evaluate(hotpot_loader, 'hotpot_relabel')

save_json(REPORT_DIR / 'synthetic_val_metrics.json', synthetic_result)
save_json(REPORT_DIR / 'hard_val_metrics.json', hard_val_result)
save_json(REPORT_DIR / 'hard_test_metrics.json', hard_test_result)
save_json(REPORT_DIR / 'v2_on_hotpot_relabel.json', hotpot_result)

summary = {
    'train_rows': len(train_df),
    'synthetic_val_rows': len(synthetic_val),
    'hard_val_rows': len(hard_val),
    'hard_test_rows': len(hard_test),
    'hotpot_relabel_rows': len(hotpot_test),
    'synthetic_val': {k: synthetic_result[k] for k in ['accuracy', 'macro_f1']},
    'hard_val': {k: hard_val_result[k] for k in ['accuracy', 'macro_f1']},
    'hard_test': {k: hard_test_result[k] for k in ['accuracy', 'macro_f1']},
    'hotpot_relabel': {k: hotpot_result[k] for k in ['accuracy', 'macro_f1']},
}
save_json(REPORT_DIR / 'summary.json', summary)

print(json.dumps(summary, indent=2))
print('hard_test confusion:', hard_test_result['confusion_matrix'])
print('hotpot confusion:', hotpot_result['confusion_matrix'])

{
  "train_rows": 1512,
  "synthetic_val_rows": 168,
  "hard_val_rows": 60,
  "hard_test_rows": 60,
  "hotpot_relabel_rows": 71,
  "synthetic_val": {
    "accuracy": 1.0,
    "macro_f1": 1.0
  },
  "hard_val": {
    "accuracy": 1.0,
    "macro_f1": 1.0
  },
  "hard_test": {
    "accuracy": 1.0,
    "macro_f1": 1.0
  },
  "hotpot_relabel": {
    "accuracy": 1.0,
    "macro_f1": 1.0
  }
}
hard_test confusion: [[20, 0, 0], [0, 20, 0], [0, 0, 20]]
hotpot confusion: [[0, 0, 0], [0, 71, 0], [0, 0, 0]]


## 8. Check Saved Model Files

In [14]:
!find models/classifier_bert_v2 -maxdepth 1 -type f -print

models/classifier_bert_v2/special_tokens_map.json
models/classifier_bert_v2/vocab.txt
models/classifier_bert_v2/tokenizer.json
models/classifier_bert_v2/tokenizer_config.json
models/classifier_bert_v2/config.json
models/classifier_bert_v2/model.safetensors


## 9. Download Outputs

In [15]:
from google.colab import files

!zip -qr /content/intent_v2_outputs.zip models/classifier_bert_v2 reports/en/intent_v2 data/en/intent_v2 data/en/intent_hard_balanced_review.csv reports/en/intent_dataset_v2_report.md
files.download('/content/intent_v2_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>